In [ ]:
import json
from dataclasses import dataclass
from typing import Literal

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from loguru import logger
from langchain.messages import HumanMessage, ToolMessage
from rich import print

import random

load_dotenv(override=True)


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    指定された都市の当日の天気を照会する

    Args:
        city:都市名
    """
    return f"{city}は晴れ、微風です"


tools = [get_weather]

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body=
    {
        "thinking": {
            "type": "disabled"
        }
    }
)

model_with_tool = model.bind_tools(tools=tools)


# 状態として Messages をそのまま使用
#1. llm ノードを定義
def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    res = model_with_tool.invoke(messages)
    return {
        "messages": [res]
    }


#2. ルーターを定義。llm_node 実行後にツール呼び出しが必要かどうかを判定
def router(state: MessagesState) -> Literal["tool_node", END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END


global_cache = dict()


def wrap_tool_call(request, execute):
    tool_name = request.tool_call["name"]
    tool_args = json.dumps(request.tool_call["args"])
    tool_call_id = request.runtime.tool_call_id

    #1. 同じ関数と引数で呼び出し済みかどうかを判定 => キャッシュを直接使用
    cache_key = (tool_name, tool_args)
    cache = global_cache.get(cache_key)

    if cache:
        # キャッシュヒット。以前に呼び出し済み
        logger.info("{} の呼び出しキャッシュがヒットしました", tool_name)
        tool_msg = ToolMessage(
            tool_call_id=tool_call_id,
            content=cache
        )
    else:
        # キャッシュがヒットしなかった。ツール関数を実行する必要がある
        tool_msg = execute(request)
        logger.info("{} の呼び出し結果をキャッシュに書き込みます", tool_name)
        global_cache[cache_key] = tool_msg.content
    return tool_msg


#3. グラフを構築
builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", ToolNode(tools=tools, wrap_tool_call=wrap_tool_call))
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, path_map=["tool_node", END])
builder.add_edge("tool_node", "llm_node")

graph = builder.compile()

from IPython.display import display

display(graph)

print("=" * 30, "1回目の呼び出し 東京 ", "=" * 30)
res = graph.invoke({"messages": [HumanMessage(content="今日の東京の天気はどうですか？")]})
print(res)
for msg in res["messages"]:
    msg.pretty_print()

In [ ]:
print("=" * 30, "2回目の呼び出し 東京 ", "=" * 30)
res2 = graph.invoke({"messages": [HumanMessage(content="今日の東京の天気はどうですか？")]})
print(res2)
for msg in res2["messages"]:
    msg.pretty_print()